# D604 Task 2 — Sentiment Analysis with a Bidirectional LSTM

**Research question:** Can a bidirectional LSTM neural network trained on combined Amazon, Yelp, and IMDb customer review data accurately classify sentiment as positive or negative across multiple review domains, to support automated customer-feedback monitoring for an online retail platform?

**Objective:** Build and train a Bidirectional LSTM model that classifies review text as positive (1) or negative (0) with strong accuracy, precision, recall, and F1 score on held-out test data, demonstrating that NLP-based sentiment classification can reliably automate the interpretation of multi-domain customer feedback at scale.

**Network type:** Bidirectional LSTM (Long Short-Term Memory) — a recurrent neural network architecture that processes text sequences in both forward and backward directions, capturing context from both sides of each word.

## Requirements

Install the following packages:

```
pip install numpy pandas matplotlib seaborn scikit-learn tensorflow nltk
```

Then run this once to download NLTK corpora:

```python
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('omw-1.4')
```

**Dataset setup:** Place the three dataset files (`Amazon_Dataset.txt`, `Yelp_Dataset.txt`, `IMDB_Dataset.txt`) in the **same folder** as this notebook, or update the paths in Section 0.

## 0. Setup and Reproducibility

In [ ]:
import os
import re
import random
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# ── Download NLTK corpora (safe to re-run) ──────────────────────────────────
for corpus in ["stopwords", "wordnet", "punkt", "punkt_tab", "omw-1.4"]:
    nltk.download(corpus, quiet=True)

# ── Dataset paths ────────────────────────────────────────────────────────────
AMAZON_PATH = "Amazon_Dataset.txt"
YELP_PATH   = "Yelp_Dataset.txt"
IMDB_PATH   = "IMDB_Dataset.txt"

# ── Hyperparameters ──────────────────────────────────────────────────────────
VOCAB_SIZE    = 5000
MAX_LEN       = 30
EMBEDDING_DIM = 64
LSTM_UNITS    = 64
DENSE_UNITS   = 32
DROPOUT_RATE  = 0.4
BATCH_SIZE    = 32
EPOCHS        = 20
SEED          = 42

# ── Reproducibility ──────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NLTK:", nltk.__version__)

## 1. Load and Combine Datasets

All three datasets (Amazon, Yelp, IMDb) are loaded and combined into a single dataframe. Each file is tab-separated with two columns: the review text and a binary label (1 = positive, 0 = negative). A `source` column is added to track origin for the EDA.

In [ ]:
dfs = []
for path, source in [
    (AMAZON_PATH, "amazon"),
    (YELP_PATH,   "yelp"),
    (IMDB_PATH,   "imdb"),
]:
    df_src = pd.read_csv(path, sep="\t", header=None,
                         names=["review", "label"], quoting=3)
    df_src["source"] = source
    dfs.append(df_src)

df = pd.concat(dfs, ignore_index=True)
print("Combined shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())
print("\nPer-source row counts:")
print(df["source"].value_counts())
df.head(6)

## 2. Exploratory Data Analysis (Part B1)

Before any preprocessing, the raw dataset is examined for: label balance, review length distribution, vocabulary size, and the presence of unusual or non-text characters such as emojis and non-ASCII characters.

In [ ]:
# ── Label distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
counts = df["label"].value_counts()
sns.barplot(x=["Negative (0)", "Positive (1)"], y=counts.values,
            hue=["Negative (0)", "Positive (1)"],
            palette=["#E24B4A", "#1D9E75"], ax=ax, legend=False)
ax.set_title("Overall Label Distribution")
ax.set_ylabel("Count")
for bar, v in zip(ax.patches, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(v), ha="center", fontsize=11)

ax = axes[1]
df.groupby(["source","label"]).size().unstack().plot(
    kind="bar", ax=ax, color=["#E24B4A","#1D9E75"], legend=True)
ax.set_title("Label Distribution by Source")
ax.set_ylabel("Count")
ax.set_xlabel("Source")
ax.legend(["Negative", "Positive"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("eda_label_distribution.png", dpi=150)
plt.show()

In [ ]:
# ── Review length distribution ──────────────────────────────────────────────
df["word_count"] = df["review"].astype(str).apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["word_count"], bins=40, kde=True, ax=ax, color="#378ADD")
ax.axvline(df["word_count"].quantile(0.95), color="#E24B4A",
           linestyle="--", label=f"95th pct = {df['word_count'].quantile(0.95):.0f} words")
ax.set_title("Review Length Distribution (word count, raw)")
ax.set_xlabel("Words per review"); ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.savefig("eda_length_distribution.png", dpi=150)
plt.show()

print(f"Mean words per review : {df['word_count'].mean():.1f}")
print(f"Median                : {df['word_count'].median():.0f}")
print(f"Max                   : {df['word_count'].max()}")
print(f"95th percentile       : {df['word_count'].quantile(0.95):.0f}")

In [ ]:
# ── Unusual / non-text characters ───────────────────────────────────────────
df["has_non_ascii"] = df["review"].astype(str).apply(
    lambda x: bool(re.search(r'[^\x00-\x7F]', x)))

n_non_ascii = df["has_non_ascii"].sum()
print(f"Reviews containing non-ASCII / emoji characters: {n_non_ascii} "
      f"({n_non_ascii/len(df)*100:.1f}%)")
print("\nExample non-ASCII reviews:")
for _, row in df[df["has_non_ascii"]].head(4).iterrows():
    chars = re.findall(r'[^\x00-\x7F]', row["review"])
    print(f"  [{row['source']}] '{row['review'][:70]}...'  →  chars: {chars}")

In [ ]:
# ── Raw vocabulary size ─────────────────────────────────────────────────────
all_words = []
for review in df["review"].astype(str):
    all_words.extend(review.lower().split())

raw_vocab = set(all_words)
print(f"Raw vocabulary size (unique tokens, lowercased): {len(raw_vocab):,}")
print(f"Total raw tokens across all reviews            : {len(all_words):,}")
print(f"Duplicate reviews detected                     : {df.duplicated(subset=['review']).sum()}")
print(f"Missing values                                 : {df.isnull().sum().to_dict()}")

## 3. Data Preprocessing (Part B2)

The preprocessing pipeline applies nine sequential noise-removal steps to prepare raw review text for tokenization. Each step is described and justified below. **All steps are applied uniformly across all three dataset sources before any train/test split**, and all preprocessing artifacts (stopword list, lemmatizer) are applied purely based on rules/dictionaries — not fit to the data — so there is no risk of data leakage.

### Preprocessing steps applied (in order):

1. **Lowercasing** — standardises all text to lowercase so "Great" and "great" are treated as the same token.
2. **Remove non-ASCII / non-English characters** — strips accented letters (e.g. é, ê), non-Latin characters, and emoji characters. These appear in 0.6% of reviews and carry no generalizable sentiment signal after encoding.
3. **Remove URLs** — strips any `http://` or `https://` patterns; not present in this dataset but defensively removed.
4. **Remove punctuation and numeric values** — replaces all non-alphabetic characters with a space. Punctuation marks (commas, exclamation marks) are redundant after lowercasing and would bloat vocabulary; numeric values ("4 stars", "10/10") convey relative rather than absolute sentiment and are inconsistent across reviews.
5. **Normalise whitespace** — collapses multiple consecutive spaces into a single space and strips leading/trailing whitespace.
6. **Remove stopwords** — removes the 179 common English stopwords from NLTK (e.g. "the", "is", "a", "and"). Stopwords carry no sentiment signal and account for a large fraction of total tokens, so removing them reduces vocabulary size and computation without information loss.
7. **Tokenize** — splits each cleaned string into a list of individual word tokens using NLTK's `word_tokenize()`.
8. **Lemmatize** — maps each token to its dictionary root form using NLTK's `WordNetLemmatizer` (e.g. "running" → "run", "movies" → "movie"). Lemmatization consolidates inflected variants of the same word into one token, reducing vocabulary size and helping the model generalise across tense and plural forms.
9. **Drop empty and duplicate rows** — removes any reviews that became empty after cleaning (insufficient content to classify) and removes exact duplicate cleaned reviews to prevent the same content appearing in both training and test sets.

In [ ]:
STOP_WORDS  = set(stopwords.words("english"))
lemmatizer  = WordNetLemmatizer()

def clean_text(text):
    text = str(text)
    # 1. Lowercase
    text = text.lower()
    # 2. Remove non-ASCII (covers accented chars, emoji, non-English glyphs)
    text = text.encode("ascii", "ignore").decode("ascii")
    # 3. Remove URLs
    text = re.sub(r'http\S+', '', text)
    # 4. Remove punctuation and numeric values (keep only a-z and spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 5. Normalise whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 6–8. Tokenize, remove stopwords, lemmatize
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOP_WORDS and len(t) > 1]
    return " ".join(tokens)

print("Applying preprocessing pipeline to all reviews...")
df["cleaned"] = df["review"].apply(clean_text)

# 9. Drop empty and duplicate rows
before = len(df)
df = df[df["cleaned"].str.len() > 0]
df = df.drop_duplicates(subset=["cleaned"])
after = len(df)

print(f"Rows before cleaning : {before}")
print(f"Rows after cleaning  : {after} (removed {before - after} empty/duplicate rows)")
print(f"\nPost-clean label balance:")
print(df["label"].value_counts())

In [ ]:
# Show side-by-side comparison of raw vs cleaned for 5 reviews
print("=" * 75)
print(f"{'ORIGINAL':<50} | CLEANED")
print("=" * 75)
for _, row in df.sample(5, random_state=SEED).iterrows():
    orig  = row["review"][:48].ljust(50)
    clean = row["cleaned"][:48]
    print(f"{orig} | {clean}")

In [ ]:
# Post-clean vocabulary size
post_words = []
for review in df["cleaned"]:
    post_words.extend(review.split())
post_vocab = set(post_words)

print(f"Post-clean vocabulary size : {len(post_vocab):,} "
      f"(down from {len(raw_vocab):,} raw tokens)")
print(f"Post-clean avg tokens/review: "
      f"{np.mean([len(r.split()) for r in df['cleaned']]):.1f}")
print(f"Post-clean 95th percentile  : "
      f"{np.percentile([len(r.split()) for r in df['cleaned']], 95):.0f} tokens")

## 4. Tokenization (Part B3)

**Goal of tokenization:** Convert cleaned text strings into sequences of integers so the neural network can process them numerically. Each unique word in the vocabulary is assigned a unique integer index; every review is then represented as a list of those integers in the order the words appear.

**Method:** Keras's `Tokenizer` is used with `num_words=VOCAB_SIZE` (5,000), which keeps only the 4,999 most frequent words plus a special `<OOV>` (out-of-vocabulary) token at index 1 for any word not in the top 5,000. The tokenizer is **fit only on the training set** to prevent data leakage — the validation and test sets are only transformed using the vocabulary learned from training data.

**Vocabulary size justification:** The post-clean vocabulary contains 4,545 unique tokens. A cap of 5,000 therefore covers the entire real vocabulary with one slot to spare, so no meaningful words are discarded as out-of-vocabulary.

In [ ]:
# Split FIRST — fit tokenizer only on training data (prevents data leakage)
X_text = df["cleaned"].values
y      = df["label"].values

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.30, random_state=SEED, stratify=y)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f"Train : {len(X_train_text):,} reviews")
print(f"Val   : {len(X_val_text):,} reviews")
print(f"Test  : {len(X_test_text):,} reviews")

# Fit tokenizer on training data only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

train_sequences = tokenizer.texts_to_sequences(X_train_text)
val_sequences   = tokenizer.texts_to_sequences(X_val_text)
test_sequences  = tokenizer.texts_to_sequences(X_test_text)

print(f"\nTokenizer vocabulary size (fit on train): "
      f"{len(tokenizer.word_index):,}")
print(f"\nExample tokenized review:")
print(f"  Cleaned text : {X_train_text[0]}")
print(f"  Integer seq  : {train_sequences[0]}")

## 5. Vectorization — Embedding Layer (Part B4)

**Goal of vectorization:** Map each integer token index to a dense, trainable vector of real numbers (an embedding). Rather than one-hot encoding (which produces a sparse 5,000-dimensional vector per token), an embedding layer learns a compact 64-dimensional representation where semantically similar words end up with similar vectors. This learned representation captures meaning from context — the model learns, for example, that "great" and "excellent" should have similar embeddings because they appear in similar contexts.

**Method:** A Keras `Embedding` layer with:
- `input_dim = VOCAB_SIZE` (5,000) — one vector per vocabulary word
- `output_dim = EMBEDDING_DIM` (64) — each word maps to a 64-dimensional vector
- Weights are randomly initialised and learned during backpropagation, not imported from a pretrained model

**Word embedding length (64):** Standard embedding sizes for small-to-medium NLP tasks range from 32 to 256. 64 is chosen as a practical middle ground — large enough to capture meaningful semantic distinctions across 5,000 vocabulary words, small enough to train efficiently on 2,000 examples without overfitting. Larger embeddings (e.g. 128 or 256) would offer marginally more representational capacity but would require substantially more data to train effectively.

**Maximum sequence length (30):** The 95th percentile of post-clean token counts is 14, and the maximum is 44. A `MAX_LEN` of 30 covers the vast majority of reviews with minimal padding waste, while keeping the LSTM computation efficient. Reviews longer than 30 tokens are truncated (the tail is cut), which discards at most a few tokens from the longest 5% of reviews — an acceptable trade-off given that most sentiment signal in short reviews is front-loaded.

In [ ]:
# Vectorization happens inside the model as an Embedding layer.
# Here we preview the embedding matrix shape as it will appear at runtime.
print(f"Embedding layer configuration:")
print(f"  input_dim  (vocabulary size)  : {VOCAB_SIZE:,}")
print(f"  output_dim (embedding length) : {EMBEDDING_DIM}")
print(f"  Total embedding parameters    : {VOCAB_SIZE * EMBEDDING_DIM:,}")
print()
print(f"Sequence length justification:")
print(f"  Post-clean 95th pct length    : "
      f"{np.percentile([len(s) for s in train_sequences], 95):.0f} tokens")
print(f"  Post-clean max length         : "
      f"{max(len(s) for s in train_sequences)} tokens")
print(f"  Selected MAX_LEN              : {MAX_LEN} "
      f"(covers >95th pct, truncates only extreme outliers)")

## 6. Sequence Padding (Part B5)

**Goal of padding:** The LSTM processes inputs in fixed-size batches, which requires every sequence to have the same length. Sequences shorter than `MAX_LEN` are **post-padded** with zeros appended at the end; sequences longer than `MAX_LEN` are **post-truncated** from the end.

**Post-padding (not pre-padding)** is chosen deliberately: the most important sentiment signal in a short review — the first few words — should arrive at the LSTM first, undiluted by a leading block of zeros. Pre-padding would force the LSTM to process many zero steps before reaching any meaningful content, potentially making it harder for the network to connect early strong sentiment words to the final classification output. The padding zeros are treated as a special token by the Embedding layer and carry no learned semantic meaning.

In [ ]:
X_train = pad_sequences(train_sequences, maxlen=MAX_LEN,
                        padding="post", truncating="post")
X_val   = pad_sequences(val_sequences,   maxlen=MAX_LEN,
                        padding="post", truncating="post")
X_test  = pad_sequences(test_sequences,  maxlen=MAX_LEN,
                        padding="post", truncating="post")

print(f"Padded array shapes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")
print()
print(f"Example padded sequence (first training review):")
print(f"  Cleaned text    : {X_train_text[0]}")
print(f"  Integer sequence: {train_sequences[0]}")
print(f"  Padded (len {MAX_LEN}) : {X_train[0]}")
print()
print(f"Note: trailing zeros are the post-padding; "
      f"meaningful tokens are at the start of each row.")

## 7. Save Combined Preprocessed Dataset (Part F2)

All preprocessing stages — noise removal, tokenization, vectorization, and padding — are combined into a single `.pkl` (pickle) file for GitLab submission.

In [ ]:
processed = {
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
    "tokenizer": tokenizer,
    "MAX_LEN": MAX_LEN,
    "VOCAB_SIZE": VOCAB_SIZE,
    "EMBEDDING_DIM": EMBEDDING_DIM,
}
with open("processed_sentiment_data.pkl", "wb") as f:
    pickle.dump(processed, f)
print("Saved processed_sentiment_data.pkl")

# To reload later:
# with open("processed_sentiment_data.pkl", "rb") as f:
#     data = pickle.load(f)
# X_train = data["X_train"]; tokenizer = data["tokenizer"]

## 8. Model Architecture (Part C)

The model is a **Bidirectional LSTM** — a recurrent architecture that reads each review sequence both left-to-right and right-to-left, then concatenates both directions' outputs. This is particularly well-suited to sentiment analysis because sentiment words (e.g. "not bad", "would not recommend") can only be fully interpreted with context from both directions: a negation before a word changes its sentiment, and a qualifying phrase after a word also matters.

### Architecture layers:

1. **Embedding** — maps integer token indices to 64-dimensional trainable word vectors. Learns during training so similar-meaning words converge to similar vector representations.
2. **Bidirectional(LSTM)** — processes the 30-step embedding sequence forward and backward, each direction with 64 LSTM units, concatenated to produce a 128-dimensional vector summarising the whole sequence. `dropout=0.3` on inputs and `recurrent_dropout=0.3` on hidden states reduces overfitting.
3. **Dense (32 units, ReLU)** — combines the LSTM's 128-dimensional representation into 32 higher-level features relevant to the positive/negative distinction.
4. **Dropout (0.4)** — randomly deactivates 40% of Dense units during training to prevent co-adaptation.
5. **Dense (1 unit, Sigmoid)** — single output unit with sigmoid activation, producing a probability in [0, 1]; values ≥ 0.5 are classified positive (1), values < 0.5 are classified negative (0).

In [ ]:
model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, name="embedding"),
    layers.Bidirectional(
        layers.LSTM(LSTM_UNITS, dropout=0.3, recurrent_dropout=0.3),
        name="bidirectional_lstm"
    ),
    layers.Dense(DENSE_UNITS, activation="relu", name="dense_hidden"),
    layers.Dropout(DROPOUT_RATE, name="dropout"),
    layers.Dense(1, activation="sigmoid", name="output"),
], name="sentiment_bilstm")

model.summary()

In [ ]:
# Activation functions per layer
print("\nLayer-by-layer breakdown:")
for layer in model.layers:
    cfg = layer.get_config()
    act = cfg.get("activation", "N/A")
    print(f"  {layer.name:25s} | {layer.__class__.__name__:18s} | "
          f"activation: {act}")

### Hyperparameter justification (Part C2)

- **Sigmoid output + Binary cross-entropy loss:** the standard pairing for binary classification. Sigmoid squashes the output to [0,1] (a probability); binary cross-entropy measures how far that probability is from the true 0/1 label.
- **Adam optimizer, lr=0.001:** adaptive per-parameter learning rates; generally converges faster than SGD on RNN tasks without extensive manual scheduling.
- **LSTM units = 64 (→ 128 bidirectional):** sufficient capacity to capture sequential sentiment dependencies in short reviews (avg 6 tokens post-clean) without overparameterising.
- **Embedding dim = 64:** balances representational richness against training data size (2,037 training samples). Larger embeddings need more data to learn meaningfully.
- **Dropout 0.3 (LSTM) + 0.4 (Dense):** standard recurrent dropout rates for small NLP tasks; LSTM dropout is applied to inputs, recurrent dropout to hidden states.
- **Early stopping, patience=5:** halts training if validation loss fails to improve for 5 epochs and restores best weights. Short patience is appropriate given the small dataset and fast convergence expected.
- **Batch size = 32:** standard; balances gradient noise (beneficial regularisation) against training stability.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=5,
    restore_best_weights=True, verbose=1
)
checkpoint = callbacks.ModelCheckpoint(
    "best_sentiment_model.keras",
    monitor="val_loss", save_best_only=True, verbose=0
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, checkpoint],
    verbose=1,
)

## 9. Model Evaluation (Part D)

In [ ]:
final_epoch = len(history.history["loss"])
best_epoch  = int(np.argmin(history.history["val_loss"])) + 1

print(f"Training ran for {final_epoch} epoch(s); best weights from epoch {best_epoch}.")
print(f"Final training accuracy   : {history.history['accuracy'][-1]:.4f}")
print(f"Final training loss       : {history.history['loss'][-1]:.4f}")
print(f"Best validation accuracy  : {history.history['val_accuracy'][best_epoch-1]:.4f}")
print(f"Best validation loss      : {history.history['val_loss'][best_epoch-1]:.4f}")

**Screenshot this output for Part D (training behavior).**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, final_epoch + 1)

axes[0].plot(ep, history.history["accuracy"],     label="Train")
axes[0].plot(ep, history.history["val_accuracy"], label="Validation")
axes[0].axvline(best_epoch, color="red", linestyle="--",
                alpha=0.6, label=f"Best (ep {best_epoch})")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(ep, history.history["loss"],     label="Train")
axes[1].plot(ep, history.history["val_loss"], label="Validation")
axes[1].axvline(best_epoch, color="red", linestyle="--",
                alpha=0.6, label=f"Best (ep {best_epoch})")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# ── Test set evaluation ──────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print(f"Test Loss      : {test_loss:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"F1 Score       : {f1:.4f}")
print()
print(classification_report(y_test, y_pred,
      target_names=["Negative (0)", "Positive (1)"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Neg", "Predicted Pos"],
            yticklabels=["True Neg", "True Pos"], ax=ax)
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (correctly predicted negative) : {tn}")
print(f"True Positives  (correctly predicted positive) : {tp}")
print(f"False Positives (negative predicted as positive): {fp}")
print(f"False Negatives (positive predicted as negative): {fn}")

In [ ]:
# ── Prediction examples ──────────────────────────────────────────────────────
print("Sample predictions on test set:")
print("-" * 70)
for i in np.random.choice(len(X_test), 8, replace=False):
    true_label = "POS" if y_test[i] == 1 else "NEG"
    pred_label = "POS" if y_pred[i] == 1 else "NEG"
    conf       = y_pred_prob[i] if y_pred[i] == 1 else 1 - y_pred_prob[i]
    match      = "✓" if true_label == pred_label else "✗"
    print(f"{match} True:{true_label} Pred:{pred_label} "
          f"({conf:.2f}) | {X_test_text[i][:55]}")

## 10. Save the Trained Model

In [ ]:
model.save("sentiment_bilstm_final_model.keras")
print("Model saved as sentiment_bilstm_final_model.keras")

## 11. AI Ethics Compliance (Part E)

This section addresses how the analysis complies with AI ethical standards — fairness, transparency, and accountability — and mitigates bias in model design and data selection.

### Fairness

**Dataset diversity:** Combining Amazon, Yelp, and IMDb reviews across three distinct domains (product, restaurant, and movie reviews) reduces domain-specific bias. A model trained on a single domain would risk learning that "delivery was fast" implies positive sentiment only in restaurant contexts, for example. The combined dataset forces the model to learn sentiment from language itself rather than domain vocabulary.

**Class balance:** The combined dataset contains exactly 1,500 positive and 1,500 negative reviews — perfectly balanced before cleaning. After cleaning, the balance is maintained (verified above). A class-balanced dataset prevents the model from achieving high accuracy simply by always predicting the majority class, and ensures precision and recall are meaningful for both classes.

**Stopword and lemmatization neutrality:** The NLTK English stopword list and WordNet lemmatizer are domain-neutral, rule-based tools that apply the same transformation to every review regardless of the reviewer's identity, origin, or writing style.

### Transparency

**Documented preprocessing:** Every preprocessing decision (which characters are removed, why MAX_LEN=30 was chosen, why post-padding was used) is explained in the notebook with both code and written justification. The notebook is fully reproducible — re-running it from scratch with the same seed will produce the same results.

**Model architecture and hyperparameters:** All architectural choices and hyperparameter values are made explicit with written rationale in Section 8, rather than treated as black-box defaults.

**Known limitations:** This model is trained on a small, curated dataset (2,911 post-clean reviews) with clearly positive or negative labels — neutral reviews, sarcasm, and culturally specific expressions of sentiment are underrepresented. Deploying this model in production without acknowledging these limitations would be misleading.

### Accountability

**No data leakage:** The tokenizer is fit only on training data and applied — not refit — to validation and test sets, ensuring reported test metrics are unbiased estimates of real-world performance.

**Reproducibility:** A fixed random seed (42) is set at the start of the notebook, and the processed dataset and trained model are both saved for independent verification and re-execution.

**Scope limitations:** The model predicts binary sentiment for English-language text reviews in consumer contexts. Using it for other languages, for high-stakes decisions (e.g. employment screening, medical feedback triage), or for nuanced opinion mining beyond positive/negative would require additional validation, fairness audits, and likely retraining on domain-specific data.

## Submission Checklist (Part F)

| File | Description |
|---|---|
| `D604_Task2_Sentiment_Analysis.ipynb` | This notebook (Part F1) |
| `processed_sentiment_data.pkl` | Combined preprocessed data — padded sequences, labels, tokenizer (Part F2) |
| `best_sentiment_model.keras` | Best checkpoint (lowest val_loss) |
| `sentiment_bilstm_final_model.keras` | Final saved model |

**Screenshots needed for the Word report (Parts B–D):**
- Example tokenized review output (Section 4)
- Example padded sequence output (Section 6)
- `model.summary()` output (Section 8)
- Final epoch metrics printout (Section 9)
- Training/validation accuracy and loss curves (Section 9)
- Confusion matrix (Section 9)
- Sample predictions table (Section 9)